In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:
df_silver=spark.table("fraud_transactions_silver")
print("عدد الصفوف",df_silver.count())
print("عدد الأعمدة",len(df_silver.columns))


In [ ]:
df_silver.groupBy("is_fraud").count().orderBy("is_fraud").show()

In [ ]:
from pyspark.sql import functions as F

gold_fraud_overview = df_silver.agg(
    F.count("*").alias("total_transactions"),

    F.sum(
        F.when(F.col("is_fraud")==0,1).otherwise (0)
    ).alias("legitimate_transactions"),
   
    F.sum(
        F.when(F.col("is_fraud")==1,1).otherwise(0)
    
    ).alias("fraud_transactions"),
    
   F.round(
    F.sum(
        F.when(F.col("is_fraud")==1,1).otherwise(0)
    )/F.count("*")*100,3
   ).alias("fraud_rate_pct"),

   F.round(
    F.sum("amt"),2
    ).alias("total_transaction_amount"),

F.round(
    F.sum(
            F.when(
            F.col("is_fraud")==1,
                F.col("amt")
                  ).otherwise(0)
        ),2
).alias("fraud_amount"),

F.round(
    F.sum(
        F.when(F.col("is_fraud")==1,F.col("amt")).otherwise(0)
    )/F.sum("amt")*100,2
).alias("fraud_amount_rate_pct")




)
display(gold_fraud_overview)

In [ ]:
(
    gold_fraud_overview.write
    .format("delta")
    .mode("errorifexists")
    .saveAsTable("gold_fraud_overview")
)

In [ ]:
saved_gold_overview = spark.table("gold_fraud_overview")

print("عدد الصفوف:", saved_gold_overview.count())
print("عدد الأعمدة:", len(saved_gold_overview.columns))

display(saved_gold_overview)

In [ ]:
df_silver=spark.table("fraud_transactions_silver")
df_silver.printSchema()

In [ ]:
from pyspark.sql import functions as F 
dim_date=(df_silver.select(
    F.to_date("trans_date_trans_time").alias("date")
)
.distinct()
.orderBy("date")

)
print("عدد الأعمدة: ",dim_date.count())
display(dim_date)

In [ ]:
dim_date=dim_date.withColumn(
    "date_key",
    F.date_format("date","yyyyMMdd").cast("int"))

display(dim_date)

In [ ]:
dim_date = (
    dim_date
    .withColumn("year",f.year("date"))
    .withColumn("quarter",f.quarter("date"))
    .withColumn("day",f.day("date"))
    )
display(dim_date)



In [ ]:
dim_date=(dim_date
.withColumn("month_number",f.month("date"))
.withColumn("month_name",f.date_format("date","MMMM"))
)

display(dim_date)

In [ ]:
dim_date = (
    dim_date
    .withColumn("day_name", f.date_format("date", "EEEE"))
)

display(dim_date)

In [ ]:
dim_date = (
    dim_date
    .withColumn(
        "day_number",
        f.dayofweek("date")
    )
)

display(dim_date)

In [ ]:
dim_date = (
    dim_date
    .withColumn(
        "week_of_year",
        f.weekofyear("date")
    )
    .withColumn(
        "year_month",
        f.date_format("date", "yyyy-MM")
    )
)

display(dim_date)

In [ ]:
total_rows = dim_date.count()

unique_date_keys = (
    dim_date
    .select("date_key")
    .distinct()
    .count()
)

print("عدد الصفوف:", total_rows)
print("عدد مفاتيح التاريخ المختلفة:", unique_date_keys)

In [ ]:
date_range = (
    dim_date
    .agg(
        f.min("date").alias("first_date"),
        f.max("date").alias("last_date")
    )
)

display(date_range)



In [ ]:
date_check = (
    dim_date
    .agg(
        f.min("date").alias("first_date"),
        f.max("date").alias("last_date"),
        f.count("*").alias("actual_days")
    )
    .withColumn(
        "expected_days",
        f.datediff(
            f.col("last_date"),
            f.col("first_date")
        ) + 1
    )
    .withColumn(
        "missing_days",
        f.col("expected_days") - f.col("actual_days")
    )
)

display(date_check)


In [ ]:
all_dates = (
    date_check
    .select(
        f.explode(
            f.sequence(
                f.col("first_date"),
                f.col("last_date"),
                f.expr("interval 1 day")
            )
        ).alias("date")
    )
)

print("عدد الأيام الكاملة:", all_dates.count())



In [ ]:
missing_dates = (
    all_dates
    .join(
        dim_date.select("date"),
        on="date",
        how="left_anti"
    )
)

display(missing_dates)



In [ ]:
dim_date = (
    all_dates
    .orderBy("date")
)
print(dim_date.count())
display(dim_date)



In [ ]:
all_dates = (
    date_check
    .select(
        f.explode(
            f.sequence(
                f.col("first_date"),
                f.col("last_date"),
                f.expr("interval 1 day")
            )
        ).alias("date")
    )
)

print("عدد الأيام الكاملة:", all_dates.count())


In [ ]:
date_check = (
    dim_date
    .agg(
        f.min("date").alias("first_date"),
        f.max("date").alias("last_date"),
        f.count("*").alias("actual_days")
    )
    .withColumn(
        "expected_days",
        f.datediff(
            f.col("last_date"),
            f.col("first_date")
        ) + 1
    )
    .withColumn(
        "missing_days",
        f.col("expected_days") - f.col("actual_days")
    )
)

display(date_check)




In [ ]:
(
    dim_date.write
    .format("delta")
    .mode("errorifexists")
    .saveAsTable("dim_date")
)

In [ ]:
saved_dim_date = spark.table("dim_date")

print("عدد الصفوف:", saved_dim_date.count())
print("عدد الأعمدة:", len(saved_dim_date.columns))

display(saved_dim_date)

In [ ]:
print("عدد الصفوف:", dim_date.count())
print("عدد الأعمدة:", len(dim_date.columns))

display(dim_date)

In [ ]:
(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dim_date")
)

In [ ]:
saved_dim_date = spark.table("dim_date")

print("عدد الصفوف:", saved_dim_date.count())
print("عدد الأعمدة:", len(saved_dim_date.columns))

In [ ]:
df_silver= spark.table("fraud_transactions_silver")
print("عدد الصفوف:", df_silver.count())
print("عدد الأعمدة:", len(df_silver.columns))

In [ ]:
from pyspark.sql import functions as F
customer_check=(df_silver.agg(
F.countDistinct("customer_id").alias("unique_customers"),
F.sum(
    F.when(
        F.col("customer_id").isNull(),1).otherwise(0)).alias("null_customer_ids")
)

)
display(customer_check)

In [ ]:
customer_profiles = (
    df_silver
    .select(
        "customer_id",
        "gender",
        "job",
        "city",
        "state",
        "zip",
        "lat",
        "long",
        "city_pop"
    )
    .distinct()
)

print("عدد ملفات العملاء المختلفة:", customer_profiles.count())

In [ ]:
dim_customer = customer_profiles

print("عدد العملاء:", dim_customer.count())
print("عدد الأعمدة:", len(dim_customer.columns))

display(dim_customer.limit(10))

In [ ]:
customer_completeness = dim_customer.summary("count")

display(customer_completeness)

In [ ]:
(
    dim_customer.write
    .format("delta")
    .mode("errorifexists")
    .saveAsTable("dim_customer")
)

In [ ]:
saved_dim_customer = spark.table("dim_customer")

print("عدد الصفوف:", saved_dim_customer.count())
print("عدد الأعمدة:", len(saved_dim_customer.columns))

display(saved_dim_customer.limit(10))

In [ ]:
from pyspark.sql import functions as F
merchant_check=(df_silver.agg(
F.countDistinct("merchant").alias("unique_merchant_names"),
    F.sum(
        F.when(
            F.col("merchant").isNull(),1).otherwise(0)
        ).alias("null_merchant_names")


    )
)
display(merchant_check)
print("merchant_check",len(merchant_check.columns))


In [ ]:
merchant_profiles = (
    df_silver
    .select(
        "merchant",
        "category"
    )
    .distinct()
)

print(
    "عدد ملفات التاجر والتصنيف المختلفة:",
    merchant_profiles.count()
)

display(merchant_profiles.limit(10))

In [ ]:
merchants_with_multiple_categories = (
    df_silver
    .groupBy("merchant")
    .agg(
        F.countDistinct("category").alias("category_count")
    )
    .filter(
        F.col("category_count") > 1
    )
    .orderBy(
        F.col("category_count").desc()
    )
)

print(
    "عدد التجار المرتبطين بأكثر من تصنيف:",
    merchants_with_multiple_categories.count()
)

display(merchants_with_multiple_categories)

In [ ]:
merchant_location_profiles = (
    df_silver
    .select(
        "merchant",
        "category",
        "merch_zipcode",
        "merch_lat",
        "merch_long"
    )
    .distinct()
)

print(
    "عدد ملفات التاجر مع الموقع:",
    merchant_location_profiles.count()
)

display(merchant_location_profiles.limit(10))

In [ ]:
merchant_completeness = merchant_profiles.summary("count")

display(merchant_completeness)

In [ ]:
dim_merchant = (
    merchant_profiles
    .withColumn(
        "merchant_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("merchant"),
                F.col("category")
            ),
            256
        )
    )
    .select(
        "merchant_key",
        "merchant",
        "category"
    )
)

print("عدد صفوف التجار:", dim_merchant.count())
print("عدد الأعمدة:", len(dim_merchant.columns))

display(dim_merchant.limit(10))

In [ ]:
merchant_key_check = (
    dim_merchant
    .agg(
        F.count("*").alias("total_rows"),
        
        F.countDistinct("merchant_key").alias("unique_keys"),
        
        F.sum(
            F.when(
                F.col("merchant_key").isNull(),
                1
            ).otherwise(0)
        ).alias("null_keys")
    )
)

display(merchant_key_check)

In [ ]:
(

    dim_merchant.write
    .format("delta")
    .mode("errorifexists")
    .saveAsTable("dim_merchant")
)



In [ ]:
saved_dim_merchant = spark.table("dim_merchant")

print("عدد الصفوف:", saved_dim_merchant.count())
print("عدد الأعمدة:", len(saved_dim_merchant.columns))

display(saved_dim_merchant.limit(10))

In [ ]:
fact_with_keys = (
    df_silver
    .withColumn(
        "date_key",
        F.date_format(
            "trans_date_trans_time",
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "merchant_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("merchant"),
                F.col("category")
            ),
            256
        )
    )
)

display(
    fact_with_keys.select(
        "trans_num",
        "date_key",
        "customer_id",
        "merchant_key"
    ).limit(10)
)

In [ ]:
fact_transactions = (
    fact_with_keys
    .select(
        "trans_num",
        "trans_date_trans_time",
        "date_key",
        "customer_id",
        "merchant_key",
        "amt",
        "is_fraud",
        "Transaction_Type",
        "Payment_Method",
        "Customer_Satisfaction_Score",
        "Loyalty_Points_Earned",
        "age_at_transaction",
        "merch_zipcode",
        "merch_lat",
        "merch_long"
    )
)

print("عدد العمليات:", fact_transactions.count())
print("عدد الأعمدة:", len(fact_transactions.columns))

display(fact_transactions.limit(10))

In [ ]:
fact_key_check = (
    fact_transactions
    .agg(
        F.count("*").alias("total_rows"),

        F.countDistinct("trans_num").alias("unique_transactions"),

        F.sum(
            F.when(F.col("trans_num").isNull(), 1).otherwise(0)
        ).alias("null_transaction_ids"),

        F.sum(
            F.when(F.col("date_key").isNull(), 1).otherwise(0)
        ).alias("null_date_keys"),

        F.sum(
            F.when(F.col("customer_id").isNull(), 1).otherwise(0)
        ).alias("null_customer_keys"),

        F.sum(
            F.when(F.col("merchant_key").isNull(), 1).otherwise(0)
        ).alias("null_merchant_keys")
    )
)

display(fact_key_check)

In [ ]:
saved_dim_date = spark.table("dim_date")
saved_dim_customer = spark.table("dim_customer")
saved_dim_merchant = spark.table("dim_merchant")

In [ ]:
missing_date_links = (
    fact_transactions
    .join(
        saved_dim_date.select("date_key"),
        on="date_key",
        how="left_anti"
    )
    .count()
)

missing_customer_links = (
    fact_transactions
    .join(
        saved_dim_customer.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

missing_merchant_links = (
    fact_transactions
    .join(
        saved_dim_merchant.select("merchant_key"),
        on="merchant_key",
        how="left_anti"
    )
    .count()
)

print("عمليات بدون تاريخ مطابق:", missing_date_links)
print("عمليات بدون عميل مطابق:", missing_customer_links)
print("عمليات بدون تاجر مطابق:", missing_merchant_links)

In [ ]:
(
    fact_transactions.write
    .format("delta")
    .mode("errorifexists")
    .saveAsTable("fact_transactions")
)

In [ ]:
saved_fact_transactions = spark.table("fact_transactions")

print("عدد الصفوف:", saved_fact_transactions.count())
print("عدد الأعمدة:", len(saved_fact_transactions.columns))

display(saved_fact_transactions.limit(10))